We would like to predict the winner of a Basketball game, as a function of the data gathered half-time.

The dataset is stored in project/data/classification/:
- The train and test inputs representing the features are stored in X_train.npy and X_test.npy respectively.
- If the home team wins, the label is 1, -1 otherwise. The train and test labels ares tored in y_train.npy and y_test.npy respectively.
Your objective is to obtain a mean accuracy superior to 0.84 on the test set.

Remark : Pay attention to the fact that the test must not be used for training. The test set should be used only once for scoring. If you compute the score several times, with different models on the test set, it means that you use it mode than one, even if you do not call a scikit model.train() method on the test set ! Note that this strict unique usage of the test set is not always common practice in companies, but try to apply it for this exercise.

You are free to choose the classification methods, but you must compare at least 2 models. You can do more than 2 but this is not mandatory for this exercise. Discuss this choice of the optimization procedures, solvers, hyperparameters, cross-validation etc. It is sufficient that 1 of your models reaches the objective score. Several methods might work, including some methods that we have not explicitely studied in the class, do no hesitate to try them.
Indication : a solution, with the correct hyperparameters, exist in scikit amon the following scikit classes :
- linear_model.LogisticRegression
- svm.SVC
- neighbors.KNeighborsClassifier
- neural_network.MLPClassifier
Please note that there is no length contraint on your solution notebook, it may be short or long.

In [1]:
import numpy as np

X_train = np.load("classification/X_train.npy")
y_train = np.load("classification/y_train.npy")

X_test = np.load("classification/X_test.npy")
y_test = np.load("classification/y_test.npy")

X_train.shape, X_test.shape

((500, 50), (500, 50))

In [2]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
from sklearn.linear_model import LogisticRegression

log_reg_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),
    param_grid={
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__solver": ["lbfgs"],
        "model__penalty": ["l2"]
    },
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

log_reg_search.fit(X_train, y_train)

log_reg_search.best_params_, log_reg_search.best_score_

/home/huelise/Epitech/hub/workshops/q-learning-2025-uvillanueva/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/huelise/Epitech/hub/workshops/q-learning-2025-uvillanueva/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/huelise/Epitech/hub/workshops/q-learning-2025-uvillanueva/.venv/lib/python3.12/site-pack

({'model__C': 0.1, 'model__penalty': 'l2', 'model__solver': 'lbfgs'},
 np.float64(0.8400000000000001))

In [4]:
from sklearn.svm import SVC

svm_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),
    param_grid=[
        {
            "model__kernel": ["linear"],
            "model__C": [0.1, 1, 10, 100]
        },
        {
            "model__kernel": ["rbf"],
            "model__C": [0.1, 1, 10, 100],
            "model__gamma": ["scale", 0.1, 0.01, 0.001]
        }
    ],
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

svm_search.fit(X_train, y_train)

svm_search.best_params_, svm_search.best_score_

({'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'},
 np.float64(0.8539999999999999))

In [5]:
model_scores = {
    "Logistic Regression": log_reg_search.best_score_,
    "SVM": svm_search.best_score_
}

best_model_name = max(model_scores, key=model_scores.get)
best_search = {
    "Logistic Regression": log_reg_search,
    "SVM": svm_search
}[best_model_name]

best_model_name, best_search.best_params_, model_scores

('SVM',
 {'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'},
 {'Logistic Regression': np.float64(0.8400000000000001),
  'SVM': np.float64(0.8539999999999999)})

In [7]:
final_model = best_search.best_estimator_
final_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.01


In [8]:
from sklearn.metrics import accuracy_score

test_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)

test_accuracy

0.856

# Interpretation - Basketball Game Outcome Prediction

## Why the methodology was changed

In the first version, cross-validation was applied to an SVM after fixing its hyperparameters.  
This is not sufficient to justify the choice of hyperparameters, because cross-validation is useful here only if it is used to compare several candidate values and select the best ones.

For that reason, the notebook now uses **GridSearchCV** on the training set for both models.  
The test set is still kept for a single final evaluation.

## Hyperparameter justification

### Logistic Regression

A `StandardScaler` is included before the classifier because the optimization of logistic regression is sensitive to feature scales.  
This also helps avoid the numerical warnings that appeared in the previous version.

The grid explores `C = [0.01, 0.1, 1, 10, 100]`.  
These values span several orders of magnitude, which is a standard and justified way to test different regularization strengths:

- small `C`: stronger regularization, simpler model
- large `C`: weaker regularization, more flexible model

The solver `lbfgs` with `l2` penalty is kept because it is a robust default for this type of classification problem.

### SVM

A `StandardScaler` is also used before the SVM because distance-based models are strongly affected by feature magnitudes.

Two kernel families are compared:

- `linear` to test whether a simple separating hyperplane is enough
- `rbf` to allow nonlinear decision boundaries

The grid explores `C = [0.1, 1, 10, 100]` to test several regularization levels.  
For the RBF kernel, `gamma` is also tuned with `['scale', 0.1, 0.01, 0.001]` in order to test how local or smooth the decision boundary should be.

## Model selection

Each model is evaluated with the same 5-fold stratified cross-validation.  
This makes the comparison fair and allows the hyperparameters to be selected from the mean validation accuracy, not from an arbitrary fixed choice.

After the comparison, the best search object is kept and its refitted estimator is used as the final model.  
Because `GridSearchCV` uses `refit=True` by default, this estimator is retrained automatically on the full training set with the selected hyperparameters before the final test evaluation.

## Conclusion

The corrected notebook now justifies the hyperparameters and uses cross-validation for an actual model-selection purpose.  
This directly addresses the remark: cross-validation is no longer applied to a single fixed configuration, but to a grid of candidate hyperparameters for each model.
